# 02 · Quality-aware calibration (component C)

Teaches a tiny model (5 numbers) to turn a similarity score into a match probability that accounts for face size and sharpness, so small, blurry faces are not unfairly rejected.

Training examples come from public LFW photos that are deliberately shrunk, blurred and darkened. It is tested on LFW people it never saw, and on TinyFace's real small faces.

**Before running:** run notebook 01 first, because this one reuses its saved TinyFace embeddings. Upload the latest `quickpic-src.zip` to `MyDrive/QuickPic/` (replace the old one), choose a T4 GPU, then Runtime → Run all. It takes about 15 minutes.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
QP = "/content/drive/MyDrive/QuickPic"

## 1. Install the code and GPU runtime

In [ ]:
!rm -rf /content/quickpic && mkdir -p /content/quickpic
!unzip -q -o "$QP/quickpic-src.zip" -d /content/quickpic
%cd /content/quickpic
!pip install -q -e . insightface scikit-image pillow-heif gdown
!pip uninstall -y -q onnxruntime && pip install -q --force-reinstall --no-deps onnxruntime-gpu

In [ ]:
import onnxruntime

providers = onnxruntime.get_available_providers()
print(providers)
assert "CUDAExecutionProvider" in providers, (
    "No GPU: Runtime > Change runtime type > T4 GPU, then rerun."
)

## 2. Download model weights and datasets

In [ ]:
!bash scripts/download_models.sh
!mkdir -p /content/lfw
![ -d /content/lfw/lfw ] || (curl -sL -o /content/lfw.tgz https://ndownloader.figshare.com/files/5976018 && tar -xzf /content/lfw.tgz -C /content/lfw)
![ -f /content/lfw/pairs.txt ] || curl -sL -o /content/lfw/pairs.txt https://ndownloader.figshare.com/files/5976006
![ -d /content/tinyface ] || (gdown -q 1xTZc7lNmWN33ECO2AKH6FycGdiqIK7W0 -O /content/tinyface.zip && unzip -q /content/tinyface.zip -d /content)
!ls /content/lfw/lfw | wc -l; ls /content/tinyface/Testing_Set/Gallery_Distractor | wc -l

## 3. Fit and test the calibrator

In [ ]:
import os
os.environ["QUICKPIC_DEVICE"] = "cuda"
!python -m experiments.calibration --lfw /content/lfw --tinyface /content/tinyface --cache "$QP/cache"

## 4. Save and summarise

In [ ]:
import glob, json, shutil
os.makedirs(f"{QP}/results", exist_ok=True)
summary = {}
for path in sorted(glob.glob("experiments/results/calibrat*.json")):
    shutil.copy(path, f"{QP}/results/")
for path in sorted(glob.glob("experiments/results/calibration_*.json")):
    record = json.load(open(path))
    r = record["results"]
    summary[r["embedder"]] = {"commit": record["commit"], "weights": r["weights"], "pairs": r["pairs"],
                              "lfw_by_height": r["lfw_test_by_height"], "tinyface": r.get("tinyface_open_set")}
print("Paste everything below back to Claude:")
print(json.dumps(summary, indent=1))